In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
packageVersion("marginaleffects")

[1] ‘0.25.1’

In [4]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "GraduationPaper/RevisetoJournal/99931-LastAuthorMergeSimilarity.csv"))
dim(data)

[1] 317960     92

In [5]:
print(names(data))

 [1] "X"                                           
 [2] "work_id"                                     
 [3] "PublishedYear"                               
 [4] "Facility"                                    
 [5] "num_fac"                                     
 [6] "paper_type"                                  
 [7] "paper_language"                              
 [8] "novel_uzzi"                                  
 [9] "novel_uzzi_bin"                              
[10] "num_fac_scientist"                           
[11] "ratio_fac_scientist"                         
[12] "bin_fac_scientist"                           
[13] "text_fac_scientist"                          
[14] "fac_scientist_team"                          
[15] "num_leader"                                  
[16] "ratio_leader"                                
[17] "bin_leader"                                  
[18] "fac_scientist_lead_num"                      
[19] "fac_scientist_lead_ratio"                    
[20] "fac_sc

In [6]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                               PublishedYear 
                                           0 
                                    Facility 
                                           0 
                                     num_fac 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                  novel_uzzi 
                                        2598 
                              novel_uzzi_bin 
                                           0 
                           num_fac_scientist 
                                           0 
                         ratio_fac_scientist 
                                           0 
                           bin_fac_scientist 
                                           0 
                          text_fac_scientist 
                                           0 
                          fac_scientist_team 
                                           0 
                                  num_leader 
                                           0 
                                ratio_leader 
                                           0 
                                  bin_leader 
                                           0 
                      fac_scientist_lead_num 
                                           0 
                    fac_scientist_lead_ratio 
                                           0 
                      fac_scientist_lead_bin 
                                           0 
                     fac_scientist_lead_text 
                                           0 
                                      CoType 
                                           0 
                        CoType_Collaboration 
                                           0 
                        CoType_Participation 
                                           0 
                              CoType_Service 
                                           0 
                                lnnum_author 
                                           0 
                                  lnnum_inst 
                                           0 
                               international 
                                           0 
                               lnnum_country 
                                           0 
                             lnnum_reference 
                                           0 
                                 open_access 
                                           0 
                                 RaoStirling 
                                           0 
                                         SDG 
                                           0 
                               lntimescited5 
                                       14707 
                              lntimescited10 
                                       12903 
                             lntimescitedall 
                                           0 
                                 lnab_length 
                                           0 
                           lnmean_career_age 
                                           0 
                                 last_author 
                                           0 
                            lnlast_avgimpact 
                                           0 
                            last_SameCountry 
                                           0 
                            last_GlobalSouth 
                                           0 
                           lnlast_insthindex 
                                           0 
                 lnlast_before_year_prod_fac 
                                         

In [7]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [8]:
# data <- data %>% filter(!is.na(mean_career_age))
# data <- data %>% filter(!is.na(frac_hype_words))
# data <- data %>% filter(!is.na(source_hindex))
# data <- data %>% filter(!is.na(open_access))
# dim(data)

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$Facility <- as.factor(data$Facility)

In [11]:
data$CoType <- factor(data$CoType)
data <- within(data, CoType <- relevel(CoType, ref = 'Service'))
data$paper_type <- factor(data$paper_type)
data <- within(data, paper_type <- relevel(paper_type, ref = 'review'))
data$text_fac_scientist <- factor(data$text_fac_scientist)
data <- within(data, text_fac_scientist <- relevel(text_fac_scientist, ref = 'NonStaffPart'))
data$fac_scientist_lead_text <- factor(data$fac_scientist_lead_text)
data <- within(data, fac_scientist_lead_text <- relevel(fac_scientist_lead_text, ref = 'NonStaffLead'))
data$open_access <- factor(data$open_access)
data <- within(data, open_access <- relevel(open_access, ref = 'False'))
data$SDG <- factor(data$SDG)
data <- within(data, SDG <- relevel(SDG, ref = 'False'))
data$last_SameCountry <- factor(data$last_SameCountry)
data <- within(data, last_SameCountry <- relevel(last_SameCountry, ref = 'NonSame'))
data$last_GlobalSouth <- factor(data$last_GlobalSouth)
data <- within(data, last_GlobalSouth <- relevel(last_GlobalSouth, ref = 'GlobalSouth'))
data$last_before_year_with_ih_bin <- factor(data$last_before_year_with_ih_bin)
data <- within(data, last_before_year_with_ih_bin <- relevel(last_before_year_with_ih_bin, ref = 'False'))
data$last_before_year_participation_bin <- factor(data$last_before_year_participation_bin)
data <- within(data, last_before_year_participation_bin <- relevel(last_before_year_participation_bin, ref = 'False'))
data$last_before_year_co_lead_bin <- factor(data$last_before_year_co_lead_bin)
data <- within(data, last_before_year_co_lead_bin <- relevel(last_before_year_co_lead_bin, ref = 'False'))
data$international <- factor(data$international)
data <- within(data, international <- relevel(international, ref = 'domestic'))

In [12]:
paper_level <- "lnnum_author + international + lnnum_reference + num_fac + SDG + lnmean_career_age"
ex_controls <- "lnlast_avgimpact + lnlast_insthindex + last_GlobalSouth + last_SameCountry + knowledge_proximity_mean"
moderating <- "lnlast_before_year_prod_fac + last_before_year_with_ih_bin"
moderating2 <- "lnlast_before_year_prod_fac + last_before_year_participation_bin"
moderating3 <- "lnlast_before_year_prod_fac + last_before_year_co_lead_bin"
disciplines <- "Agricultural.and.Biological.Sciences + Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [13]:
paper_vars <- c("lnnum_author", "international", "lnnum_reference", "num_fac", "SDG", "lnmean_career_age")
ex_vars <- c("lnlast_avgimpact", "lnlast_insthindex", "last_GlobalSouth", "last_SameCountry", "knowledge_proximity_mean")
moderating_var <- c("lnlast_before_year_prod_fac", "last_before_year_with_ih_bin")
moderating2_var <- c("lnlast_before_year_prod_fac", "last_before_year_participation_bin")
moderating3_var <- c("lnlast_before_year_prod_fac", "last_before_year_co_lead_bin")
disciplines_vars <- c("Agricultural.and.Biological.Sciences", "Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

# H1:With > Without

In [14]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_total_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_total_bin)

NOTE: 42,366 observations removed because of NA values (LHS: 42,366, RHS: 42,366, Fixed-effects: 42,366).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 273,125
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
text_fac_scientistStaffPart                   0.081315   0.010400   7.819061
lnnum_author                                 -0.125868   0.007499 -16.785508
internationalinternational                   -0.046838   0.009772  -4.792872
lnnum_reference                               0.056831   0.009042   6.284937
num_fac                                       0.076664   0.006908  11.097094
SDGTrue                                       0.075029   0.008520   8.806287
lnmean_career_age                            -0.017545   0.013446  -1.304821
lnlast_avgimpact                             -0.244643   0.006355 -38.496044
lnlast_insthindex                            -0.026812   0.005933  -4.518741
last_GlobalSouthGlobalNorth                   0.305341   0.017870  17.

In [15]:
# 每组 reg_class 的平均预测概率
pred_bin <- avg_predictions(model_total_bin, variables = "text_fac_scientist")
pred_bin

text_fac_scientist,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NonStaffPart,0.3465627,0.02960296,11.70703,1.173205e-31,102.7493,0.2885419,0.4045834
StaffPart,0.3637671,0.03025959,12.02155,2.737939e-33,108.1705,0.3044594,0.4230748


In [16]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_pred.csv")
# write.csv(pred_bin, fname, row.names = FALSE)

In [17]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                & 0.081$^{***}$\\   
                                                & (0.010)\\   
   lnnum\_author                                & -0.126$^{***}$\\   
                                                & (0.007)\\   
   internationalinternational                   & -0.047$^{***}$\\   
                                                & (0.010)\\   
   lnnum\_reference                             & 0.057$^{***}$\\   
                                                & (0.009)\\   
   num\_fac                                     & 0.077$^{***}$\\   
                                                & (0.007)\\   
   SDGTrue                                      & 0.075$^{***}$\\   
                        

In [18]:
margins_eff_bin <- avg_comparisons(model_total_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.049643,0.006907395,151.9593,0,Inf,1.036105,1.063181,0.261085,0.277074,0.261085


In [19]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_comp_ratio.csv")
# write.csv(margins_eff_bin, fname, row.names = FALSE)

# H1 different disciplines

In [20]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ps_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ps_bin)

NOTE: 34,712 observations removed because of NA values (LHS: 34,712, RHS: 34,712, Fixed-effects: 34,712).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 236,379
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
text_fac_scientistStaffPart                   0.070454   0.010947   6.435783
lnnum_author                                 -0.202150   0.008420 -24.009627
internationalinternational                   -0.080801   0.010669  -7.573755
lnnum_reference                               0.074842   0.009614   7.784900
num_fac                                       0.112658   0.007418  15.186707
SDGTrue                                       0.025031   0.009293   2.693466
lnmean_career_age                            -0.065049   0.014561  -4.467446
lnlast_avgimpact                             -0.185837   0.007005 -26.530700
lnlast_insthindex                            -0.012331   0.006467  -1.906658
last_GlobalSouthGlobalNorth                   0.292991   0.019176  15.

In [21]:
margins_eff_ps_bin <- avg_comparisons(model_ps_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_ps_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.043481,0.007211544,144.6959,0,Inf,1.029347,1.057615,0.2640665,0.2779838,0.2640665


In [22]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_ps_comp_ratio.csv")
# write.csv(margins_eff_ps_bin, fname, row.names = FALSE)

In [23]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ps_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                & 0.070$^{***}$\\   
                                                & (0.011)\\   
   lnnum\_author                                & -0.202$^{***}$\\   
                                                & (0.008)\\   
   internationalinternational                   & -0.081$^{***}$\\   
                                                & (0.011)\\   
   lnnum\_reference                             & 0.075$^{***}$\\   
                                                & (0.010)\\   
   num\_fac                                     & 0.113$^{***}$\\   
                                                & (0.007)\\   
   SDGTrue                                      & 0.025$^{***}$\\   
                        

In [24]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ls_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ls_bin)

NOTES: 13,892 observations removed because of NA values (LHS: 13,892, RHS: 13,892, Fixed-effects: 13,892).
       2 fixed-effects (4 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 70,337
Fixed-effects: PublishedYear: 46
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
text_fac_scientistStaffPart                   0.113046   0.023331   4.845399
lnnum_author                                  0.180385   0.017241  10.462383
internationalinternational                    0.111894   0.018150   6.165011
lnnum_reference                               0.145338   0.018785   7.736957
num_fac                                      -0.060284   0.013161  -4.580447
SDGTrue                                       0.137956   0.015996   8.624487
lnmean_career_age                             0.098343   0.025046   3.926456
lnlast_avgimpact                             -0.438829   0.012450 -35.248358
lnlast_insthindex                            -0.021464   0.011375  -1.886897
last_GlobalSouthGlobalNorth                   0.115078   0.038683   2.9

In [25]:
margins_eff_ls_bin <- avg_comparisons(model_ls_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_ls_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.056879,0.01393498,75.84356,0,Inf,1.029567,1.084191,0.3070609,0.3316239,0.3070609


In [26]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_ls_comp_ratio.csv")
# write.csv(margins_eff_ls_bin, fname, row.names = FALSE)

In [27]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ls_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                & 0.113$^{***}$\\   
                                                & (0.023)\\   
   lnnum\_author                                & 0.180$^{***}$\\   
                                                & (0.017)\\   
   internationalinternational                   & 0.112$^{***}$\\   
                                                & (0.018)\\   
   lnnum\_reference                             & 0.145$^{***}$\\   
                                                & (0.019)\\   
   num\_fac                                     & -0.060$^{***}$\\   
                                                & (0.013)\\   
   SDGTrue                                      & 0.138$^{***}$\\   
                         

In [28]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_hs_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_hs_bin)

NOTES: 8,210 observations removed because of NA values (LHS: 8,210, RHS: 8,210, Fixed-effects: 8,210).
       7 fixed-effects (10 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 28,986
Fixed-effects: PublishedYear: 38
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error     z value
text_fac_scientistStaffPart                    0.349758   0.040071    8.728414
lnnum_author                                   0.238587   0.027792    8.584726
internationalinternational                     0.073157   0.028046    2.608475
lnnum_reference                                0.145033   0.029727    4.878896
num_fac                                       -0.099153   0.022155   -4.475346
SDGTrue                                        0.210391   0.025213    8.344474
lnmean_career_age                              0.166853   0.040361    4.134024
lnlast_avgimpact                              -0.357707   0.019423  -18.416711
lnlast_insthindex                             -0.022833   0.017396   -1.312529
last_GlobalSouthGlobalNorth                    0.29

In [29]:
margins_eff_hs_bin <- avg_comparisons(model_hs_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_hs_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.148676,0.03742427,30.69336,6.980536e-207,684.8358,1.075326,1.222027,0.6688488,0.7413009,0.6688488


In [30]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_hs_comp_ratio.csv")
# write.csv(margins_eff_hs_bin, fname, row.names = FALSE)

In [31]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_hs_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                & 0.350$^{***}$\\   
                                                & (0.040)\\   
   lnnum\_author                                & 0.239$^{***}$\\   
                                                & (0.028)\\   
   internationalinternational                   & 0.073$^{***}$\\   
                                                & (0.028)\\   
   lnnum\_reference                             & 0.145$^{***}$\\   
                                                & (0.030)\\   
   num\_fac                                     & -0.099$^{***}$\\   
                                                & (0.022)\\   
   SDGTrue                                      & 0.210$^{***}$\\   
                         

In [32]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_nps_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ], family = binomial("logit"), vcov = "hetero")
summary(model_nps_bin)

NOTES: 7,654 observations removed because of NA values (LHS: 7,654, RHS: 7,654, Fixed-effects: 7,654).
       6 fixed-effects (11 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 36,735
Fixed-effects: PublishedYear: 38
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error     z value
text_fac_scientistStaffPart                    0.299875   0.038622    7.764310
lnnum_author                                   0.260322   0.024324   10.702261
internationalinternational                     0.132678   0.025455    5.212175
lnnum_reference                               -0.048035   0.028696   -1.673936
num_fac                                       -0.117326   0.019919   -5.890094
SDGTrue                                        0.251293   0.022779   11.031759
lnmean_career_age                              0.148785   0.036698    4.054309
lnlast_avgimpact                              -0.559184   0.017659  -31.666369
lnlast_insthindex                             -0.057632   0.016065   -3.587425
last_GlobalSouthGlobalNorth                    0.24

In [33]:
margins_eff_nps_bin <- avg_comparisons(model_nps_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_nps_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.15914,0.03564953,32.51488,6.571103e-232,767.9712,1.089268,1.229012,0.6504019,0.715181,0.6504019


In [34]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_nps_comp_ratio.csv")
# write.csv(margins_eff_nps_bin, fname, row.names = FALSE)

In [35]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_nps_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                & 0.300$^{***}$\\   
                                                & (0.039)\\   
   lnnum\_author                                & 0.260$^{***}$\\   
                                                & (0.024)\\   
   internationalinternational                   & 0.133$^{***}$\\   
                                                & (0.025)\\   
   lnnum\_reference                             & -0.048$^{*}$\\   
                                                & (0.029)\\   
   num\_fac                                     & -0.117$^{***}$\\   
                                                & (0.020)\\   
   SDGTrue                                      & 0.251$^{***}$\\   
                          

# H2: Collaboration > Participation

In [36]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_total <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_total)

NOTE: 42,366 observations removed because of NA values (LHS: 42,366, RHS: 42,366, Fixed-effects: 42,366).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 273,125
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.112047   0.011528   9.719655
CoTypeParticipation                           0.005402   0.015999   0.337646
lnnum_author                                 -0.119416   0.007609 -15.694138
internationalinternational                   -0.046719   0.009770  -4.781675
lnnum_reference                               0.057373   0.009042   6.344928
num_fac                                       0.076336   0.006911  11.045608
SDGTrue                                       0.074198   0.008522   8.706704
lnmean_career_age                            -0.019820   0.013453  -1.473350
lnlast_avgimpact                             -0.244736   0.006356 -38.501853
lnlast_insthindex                            -0.024371   0.005949  -4.

In [37]:
# 每组 reg_class 的平均预测概率
pred <- avg_predictions(model_total, variables = "CoType")
pred

CoType,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Service,0.3466111,0.02960925,11.70618,1.185056e-31,102.7348,0.2885780,0.4046441
Collaboration,0.3704161,0.03052186,12.13609,6.799831e-34,110.1801,0.3105943,0.4302378
Participation,0.3477429,0.02975156,11.68822,1.464214e-31,102.4296,0.2894309,0.4060548


In [38]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_pred.csv")
# write.csv(pred, fname, row.names = FALSE)

In [39]:
# 每组 reg_class 的平均预测概率
margins_eff <- avg_comparisons(model_total, variables = "CoType", comparison = 'ratio')
margins_eff

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.068679,0.007978700,133.9415,0,Inf,1.0530413,1.084317,0.2606975,0.2828639,0.2606975
CoType,mean(Participation) / mean(Service),1.003265,0.009682836,103.6128,0,Inf,0.9842873,1.022243,0.2606975,0.2617400,0.2606975


In [40]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_comp_ratio.csv")
# write.csv(margins_eff, fname, row.names = FALSE)

In [41]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                          & 0.112$^{***}$\\   
                                                & (0.011)\\   
   CoTypeParticipation                          & 0.005\\   
                                                & (0.016)\\   
   lnnum\_author                                & -0.119$^{***}$\\   
                                                & (0.008)\\   
   internationalinternational                   & -0.047$^{***}$\\   
                                                & (0.010)\\   
   lnnum\_reference                             & 0.057$^{***}$\\   
                                                & (0.009)\\   
   num\_fac                                     & 0.076$^{***}$\\   
                                

# H2 Discipline

In [42]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ps <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ps)

NOTE: 34,712 observations removed because of NA values (LHS: 34,712, RHS: 34,712, Fixed-effects: 34,712).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 236,379
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.095254   0.012083   7.883175
CoTypeParticipation                           0.008066   0.016803   0.480042
lnnum_author                                 -0.196343   0.008538 -22.995645
internationalinternational                   -0.080574   0.010666  -7.554057
lnnum_reference                               0.075198   0.009613   7.822398
num_fac                                       0.112361   0.007421  15.141508
SDGTrue                                       0.024395   0.009295   2.624462
lnmean_career_age                            -0.066916   0.014567  -4.593528
lnlast_avgimpact                             -0.185953   0.007005 -26.544320
lnlast_insthindex                            -0.010205   0.006484  -1.

In [43]:
# 每组 reg_class 的平均预测概率
margins_eff_ps <- avg_comparisons(model_ps, variables = "CoType", comparison = 'ratio')
margins_eff_ps

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.059000,0.008204009,129.08323,0,Inf,1.0429204,1.075080,0.2636921,0.2825977,0.2636921
CoType,mean(Participation) / mean(Service),1.004934,0.010299339,97.57271,0,Inf,0.9847481,1.025121,0.2636921,0.2652612,0.2636921


In [44]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_ps_comp_ratio.csv")
# write.csv(margins_eff_ps, fname, row.names = FALSE)

In [45]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ps,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                          & 0.095$^{***}$\\   
                                                & (0.012)\\   
   CoTypeParticipation                          & 0.008\\   
                                                & (0.017)\\   
   lnnum\_author                                & -0.196$^{***}$\\   
                                                & (0.009)\\   
   internationalinternational                   & -0.081$^{***}$\\   
                                                & (0.011)\\   
   lnnum\_reference                             & 0.075$^{***}$\\   
                                                & (0.010)\\   
   num\_fac                                     & 0.112$^{***}$\\   
                                

In [46]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ls <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ls)

NOTES: 13,892 observations removed because of NA values (LHS: 13,892, RHS: 13,892, Fixed-effects: 13,892).
       2 fixed-effects (4 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 70,337
Fixed-effects: PublishedYear: 46
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.096800   0.025855   3.744029
CoTypeParticipation                           0.160543   0.040047   4.008877
lnnum_author                                  0.179939   0.017242  10.435870
internationalinternational                    0.111468   0.018152   6.140731
lnnum_reference                               0.145096   0.018789   7.722562
num_fac                                      -0.060130   0.013164  -4.567918
SDGTrue                                       0.138025   0.015997   8.628436
lnmean_career_age                             0.099609   0.025060   3.974727
lnlast_avgimpact                             -0.438806   0.012450 -35.246619
lnlast_insthindex                            -0.022122   0.011386  -1.9

In [47]:
# 每组 reg_class 的平均预测概率
margins_eff_ls <- avg_comparisons(model_ls, variables = "CoType", comparison = 'ratio')
margins_eff_ls

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.048688,0.01459297,71.86255,0,Inf,1.020086,1.077289,0.3072144,0.3281919,0.3072144
CoType,mean(Participation) / mean(Service),1.080859,0.02263597,47.74961,0,Inf,1.036493,1.125225,0.3072144,0.3423967,0.3072144


In [48]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_ls_comp_ratio.csv")
# write.csv(margins_eff_ls, fname, row.names = FALSE)

In [49]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ls,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                          & 0.097$^{***}$\\   
                                                & (0.026)\\   
   CoTypeParticipation                          & 0.161$^{***}$\\   
                                                & (0.040)\\   
   lnnum\_author                                & 0.180$^{***}$\\   
                                                & (0.017)\\   
   internationalinternational                   & 0.111$^{***}$\\   
                                                & (0.018)\\   
   lnnum\_reference                             & 0.145$^{***}$\\   
                                                & (0.019)\\   
   num\_fac                                     & -0.060$^{***}$\\   
                         

In [50]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_hs <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_hs)

NOTES: 8,210 observations removed because of NA values (LHS: 8,210, RHS: 8,210, Fixed-effects: 8,210).
       7 fixed-effects (10 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 28,986
Fixed-effects: PublishedYear: 38
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error     z value
CoTypeCollaboration                            0.318117   0.045487    6.993570
CoTypeParticipation                            0.431170   0.067770    6.362257
lnnum_author                                   0.237376   0.027779    8.545202
internationalinternational                     0.072265   0.028051    2.576176
lnnum_reference                                0.145028   0.029733    4.877623
num_fac                                       -0.099467   0.022166   -4.487425
SDGTrue                                        0.210641   0.025216    8.353553
lnmean_career_age                              0.168547   0.040380    4.173998
lnlast_avgimpact                              -0.357615   0.019422  -18.413213
lnlast_insthindex                             -0.02

In [51]:
# 每组 reg_class 的平均预测概率
margins_eff_hs <- avg_comparisons(model_hs, variables = "CoType", comparison = 'ratio')
margins_eff_hs

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.135532,0.03583460,31.68815,2.262774e-220,729.6461,1.065298,1.205767,0.6688044,0.7351478,0.6688044
CoType,mean(Participation) / mean(Service),1.182161,0.05014371,23.57546,6.883519e-123,405.8140,1.083881,1.280441,0.6688044,0.7565675,0.6688044


In [52]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_hs_comp_ratio.csv")
# write.csv(margins_eff_hs, fname, row.names = FALSE)

In [53]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_hs,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                          & 0.318$^{***}$\\   
                                                & (0.045)\\   
   CoTypeParticipation                          & 0.431$^{***}$\\   
                                                & (0.068)\\   
   lnnum\_author                                & 0.237$^{***}$\\   
                                                & (0.028)\\   
   internationalinternational                   & 0.072$^{***}$\\   
                                                & (0.028)\\   
   lnnum\_reference                             & 0.145$^{***}$\\   
                                                & (0.030)\\   
   num\_fac                                     & -0.100$^{***}$\\   
                         

In [54]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_nps <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ], family = binomial("logit"), vcov = "hetero")
summary(model_nps)

NOTES: 7,654 observations removed because of NA values (LHS: 7,654, RHS: 7,654, Fixed-effects: 7,654).
       6 fixed-effects (11 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 36,735
Fixed-effects: PublishedYear: 38
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error     z value
CoTypeCollaboration                            0.259465   0.044678    5.807505
CoTypeParticipation                            0.395495   0.063833    6.195790
lnnum_author                                   0.259847   0.024321   10.683913
internationalinternational                     0.132649   0.025455    5.211022
lnnum_reference                               -0.048749   0.028703   -1.698391
num_fac                                       -0.117200   0.019928   -5.881031
SDGTrue                                        0.251700   0.022783   11.047773
lnmean_career_age                              0.150690   0.036712    4.104594
lnlast_avgimpact                              -0.559250   0.017657  -31.672244
lnlast_insthindex                             -0.05

In [55]:
# 每组 reg_class 的平均预测概率
margins_eff_nps <- avg_comparisons(model_nps, variables = "CoType", comparison = 'ratio')
margins_eff_nps

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.137504,0.03461788,32.85887,8.509612e-237,784.2079,1.069654,1.205354,0.6506765,0.7071286,0.6506765
CoType,mean(Participation) / mean(Service),1.210474,0.05194766,23.30180,4.250477e-120,396.5437,1.108659,1.312290,0.6506765,0.7344871,0.6506765


In [56]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_nps_comp_ratio.csv")
# write.csv(margins_eff_nps, fname, row.names = FALSE)

In [57]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_nps,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                          & 0.259$^{***}$\\   
                                                & (0.045)\\   
   CoTypeParticipation                          & 0.395$^{***}$\\   
                                                & (0.064)\\   
   lnnum\_author                                & 0.260$^{***}$\\   
                                                & (0.024)\\   
   internationalinternational                   & 0.133$^{***}$\\   
                                                & (0.025)\\   
   lnnum\_reference                             & -0.049$^{*}$\\   
                                                & (0.029)\\   
   num\_fac                                     & -0.117$^{***}$\\   
                          

# H3: Too much will suppress

# H3a: Participation too much not good

In [58]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ ratio_fac_scientist + I(ratio_fac_scientist^2) +  ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_h3_pratio <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_h3_pratio)

NOTE: 42,366 observations removed because of NA values (LHS: 42,366, RHS: 42,366, Fixed-effects: 42,366).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 273,125
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
ratio_fac_scientist                           0.296426   0.059327   4.996507
I(ratio_fac_scientist^2)                     -0.146006   0.071108  -2.053317
lnnum_author                                 -0.117575   0.007488 -15.700843
internationalinternational                   -0.043629   0.009817  -4.444134
lnnum_reference                               0.058428   0.009049   6.457116
num_fac                                       0.076445   0.006922  11.044099
SDGTrue                                       0.074110   0.008522   8.696590
lnmean_career_age                            -0.017258   0.013449  -1.283257
lnlast_avgimpact                             -0.244609   0.006356 -38.484626
lnlast_insthindex                            -0.024399   0.005970  -4.

In [59]:
# library(marginaleffects)
# # 设置 draw = FALSE，直接拦截绘图数据
# plot_data <- plot_predictions(model_h3_pratio, condition = "ratio_fac_scientist", draw = FALSE)
# # 选出我们最需要的几列：x轴变量、预测值(estimate)、置信区间下限(conf.low)、上限(conf.high)
# export_data <- plot_data[, c("ratio_fac_scientist", "estimate", "conf.low", "conf.high")]
# # # 导出为 CSV 文件，给 Python 准备
# write.csv(export_data, "R_ex_ld_h3_pred_pratio.csv", row.names = FALSE)
# export_data
# # print("数据已成功导出！")
# # head(export_data)

In [60]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_h3_pratio,
                           keep = c("ratio_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   ratio\_fac\_scientist                        & 0.296$^{***}$\\   
                                                & (0.059)\\   
   ratio\_fac\_scientist square                 & -0.146$^{**}$\\   
                                                & (0.071)\\   
   lnnum\_author                                & -0.118$^{***}$\\   
                                                & (0.007)\\   
   internationalinternational                   & -0.044$^{***}$\\   
                                                & (0.010)\\   
   lnnum\_reference                             & 0.058$^{***}$\\   
                                                & (0.009)\\   
   num\_fac                                     & 0.076$^{***}$\\   
                        

# H3b: Lead too much not good

In [61]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ fac_scientist_lead_ratio + I(fac_scientist_lead_ratio^2) +  ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_h3_lratio <- feglm(fml, data = data[(data$paper_type =='article')&(data$CoType_Service==0)&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_h3_lratio)

NOTE: 11,714 observations removed because of NA values (LHS: 11,714, RHS: 11,714, Fixed-effects: 11,714).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 79,026
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
fac_scientist_lead_ratio                      0.155046   0.085590   1.811493
I(fac_scientist_lead_ratio^2)                -0.115490   0.087596  -1.318435
lnnum_author                                 -0.292740   0.014790 -19.793746
internationalinternational                   -0.094049   0.020934  -4.492646
lnnum_reference                               0.121685   0.017538   6.938450
num_fac                                       0.110344   0.010491  10.518264
SDGTrue                                       0.079599   0.016496   4.825440
lnmean_career_age                            -0.156374   0.029119  -5.370200
lnlast_avgimpact                             -0.160841   0.012664 -12.700495
lnlast_insthindex                            -0.057822   0.011084  -5.2

In [62]:
# library(marginaleffects)
# # 设置 draw = FALSE，直接拦截绘图数据
# plot_data <- plot_predictions(model_h3_lratio, condition = "fac_scientist_lead_ratio", draw = FALSE)
# # 选出我们最需要的几列：x轴变量、预测值(estimate)、置信区间下限(conf.low)、上限(conf.high)
# export_data <- plot_data[, c("fac_scientist_lead_ratio", "estimate", "conf.low", "conf.high")]
# # # 导出为 CSV 文件，给 Python 准备
# write.csv(export_data, "R_ex_ld_h3_pred_lratio.csv", row.names = FALSE)
# export_data
# # print("数据已成功导出！")
# # head(export_data)

In [63]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_h3_lratio,
                           keep = c("fac_scientist_lead_ratio", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                          & novel\_uzzi\_bin\\    
   Model:                                       & (1)\\  
   \midrule
   \emph{Variables}\\
   fac\_scientist\_lead\_ratio                  & 0.155$^{*}$\\   
                                                & (0.086)\\   
   fac\_scientist\_lead\_ratio square           & -0.115\\   
                                                & (0.088)\\   
   lnnum\_author                                & -0.293$^{***}$\\   
                                                & (0.015)\\   
   internationalinternational                   & -0.094$^{***}$\\   
                                                & (0.021)\\   
   lnnum\_reference                             & 0.122$^{***}$\\   
                                                & (0.018)\\   
   num\_fac                                     & 0.110$^{***}$\\   
                                 

# Moderating

In [64]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*lnlast_before_year_prod_fac  + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_pre_facpub <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_facpub)

NOTE: 42,366 observations removed because of NA values (LHS: 42,366, RHS: 42,366, Fixed-effects: 42,366).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 273,125
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                 Estimate Std. Error    z value
CoTypeCollaboration                              0.076306   0.024685   3.091205
CoTypeParticipation                              0.117481   0.033229   3.535496
lnlast_before_year_prod_fac                     -0.042988   0.004662  -9.221010
lnnum_author                                    -0.118879   0.007626 -15.589392
internationalinternational                      -0.046631   0.009771  -4.772249
lnnum_reference                                  0.057209   0.009042   6.327173
num_fac                                          0.076996   0.006929  11.111954
SDGTrue                                          0.074033   0.008523   8.686566
lnmean_career_age                               -0.019992   0.013453  -1.486101
lnlast_avgimpact                        

In [65]:
# # 1. 找到你这个连续变量的实际最小值和最大值（假设是 0 和 10，你需要改成你的实际极值）
# min_val <- min(data$lnex_ld_avg_before_year_prod_fac, na.rm=TRUE)
# max_val <- max(data$lnex_ld_avg_before_year_prod_fac, na.rm=TRUE)
# # 2. 运行估计
# res_pre_facpub <- avg_comparisons(
#     model_pre_facpub,
#     variables = "CoType",
#     comparison = "ratio",
#     newdata = datagrid(
#     model = model_pre_facpub,
#     lnex_ld_avg_before_year_prod_fac = seq(min_val, max_val, length.out = 50) # 这里的 10 可以改成任意你想要的数字
#   ),
#     by = "lnex_ld_avg_before_year_prod_fac"
# )
# res_pre_facpub

In [66]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_facpub.csv")
# write.csv(res_pre_facpub, fname, row.names = FALSE)

In [67]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_facpub,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                               & novel\_uzzi\_bin\\    
   Model:                                                            & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                               & 0.076$^{***}$\\   
                                                                     & (0.025)\\   
   CoTypeParticipation                                               & 0.117$^{***}$\\   
                                                                     & (0.033)\\   
   lnlast\_before\_year\_prod\_fac                                   & -0.043$^{***}$\\   
                                                                     & (0.005)\\   
   lnnum\_author                                                     & -0.119$^{***}$\\   
                                                                     & (0.008)\\   
   internationa

In [68]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*last_before_year_with_ih_bin  + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_pre_withih <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_withih)

NOTE: 42,366 observations removed because of NA values (LHS: 42,366, RHS: 42,366, Fixed-effects: 42,366).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 273,125
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                      Estimate Std. Error
CoTypeCollaboration                                   0.249419   0.026883
CoTypeParticipation                                   0.289234   0.034839
last_before_year_with_ih_binTrue                      0.244757   0.011864
lnnum_author                                         -0.117487   0.007632
internationalinternational                           -0.048562   0.009772
lnnum_reference                                       0.057623   0.009043
num_fac                                               0.077171   0.006907
SDGTrue                                               0.073949   0.008524
lnmean_career_age                                    -0.021512   0.013455
lnlast_avgimpact                                     -0.245370   0.006358
lnlast_insthindex         

In [70]:
# 每组 reg_class 的平均预测概率
# res_pre_withih <- avg_comparisons(model_pre_withih, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_with_ih_bin')
# res_pre_withih

In [71]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_withih.csv")
# write.csv(res_pre_withih, fname, row.names = FALSE)

In [72]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_withih,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                      & novel\_uzzi\_bin\\    
   Model:                                                                   & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                      & 0.249$^{***}$\\   
                                                                            & (0.027)\\   
   CoTypeParticipation                                                      & 0.289$^{***}$\\   
                                                                            & (0.035)\\   
   last\_before\_year\_with\_ih\_binTrue                                    & 0.245$^{***}$\\   
                                                                            & (0.012)\\   
   lnnum\_author                                                            & -0.117$^{***}$\\   
                                     

In [73]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*last_before_year_participation_bin  + ", paper_level, "+", ex_controls, "+", moderating2, "+",disciplines, " | PublishedYear")
)
model_pre_partic <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_partic)

NOTE: 42,366 observations removed because of NA values (LHS: 42,366, RHS: 42,366, Fixed-effects: 42,366).



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 273,125
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                            Estimate Std. Error
CoTypeCollaboration                                         0.186346   0.015898
CoTypeParticipation                                         0.213766   0.026615
last_before_year_participation_binTrue                      0.163061   0.012486
lnnum_author                                               -0.119524   0.007653
internationalinternational                                 -0.047122   0.009774
lnnum_reference                                             0.058090   0.009039
num_fac                                                     0.075449   0.006914
SDGTrue                                                     0.074118   0.008521
lnmean_career_age                                          -0.017495   0.013452
lnlast_avgimpact                        

In [ ]:
# 每组 reg_class 的平均预测概率
res_pre_partic <- avg_comparisons(model_pre_partic, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_participation_bin')
res_pre_partic

In [ ]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_partic.csv")
# write.csv(res_pre_partic, fname, row.names = FALSE)

In [ ]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_partic,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

In [ ]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*last_before_year_co_lead_bin  + ", paper_level, "+", ex_controls, "+", moderating3, "+",disciplines, " | PublishedYear")
)
model_pre_co_lead <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_co_lead)

In [ ]:
# 每组 reg_class 的平均预测概率
res_pre_co_lead <- avg_comparisons(model_pre_co_lead, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_co_lead_bin')
res_pre_co_lead

In [ ]:
# fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_co_lead.csv")
# write.csv(res_pre_co_lead, fname, row.names = FALSE)

In [ ]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_co_lead,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

# 补充一个更deep的point，曾经开展过“Co-lead”,后续合作/参与的收益受损更严重

In [88]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", disciplines, " | PublishedYear")
)
model1 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model1)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.168124   0.010601  15.859520
CoTypeParticipation                           0.047151   0.014212   3.317634
Arts.and.Humanities                           1.837644   0.072240  25.438148
Biochemistry..Genetics.and.Molecular.Biology  0.498277   0.012207  40.818163
Business..Management.and.Accounting          -0.428431   0.087633  -4.888910
Chemical.Engineering                          0.131064   0.020722   6.324869
Chemistry                                     0.458689   0.010925  41.986380
Computer.Science                              0.896830   0.039192  22.883139
Decision.Sciences                             1.233745   0.187252   6.588695
Dentistry                                     1.993278   0.134826  14.

In [89]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+",disciplines, " | PublishedYear")
)
model2 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model2)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.176029   0.010963  16.055991
CoTypeParticipation                           0.095087   0.014820   6.416025
lnnum_author                                 -0.130941   0.007103 -18.433963
internationalinternational                   -0.007958   0.008684  -0.916399
lnnum_reference                              -0.005923   0.008397  -0.705350
num_fac                                       0.050942   0.006548   7.779950
SDGTrue                                       0.087651   0.008086  10.840337
lnmean_career_age                            -0.007336   0.012674  -0.578814
Arts.and.Humanities                           1.805187   0.072180  25.009688
Biochemistry..Genetics.and.Molecular.Biology  0.488323   0.012268  39.

In [90]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+",disciplines, " | PublishedYear")
)
model3 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model3)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.143687   0.011097  12.948869
CoTypeParticipation                           0.047011   0.014969   3.140640
lnnum_author                                 -0.068658   0.007116  -9.648419
internationalinternational                   -0.031089   0.009559  -3.252299
lnnum_reference                               0.061520   0.008603   7.150632
num_fac                                       0.059498   0.006605   9.007958
SDGTrue                                       0.092371   0.008114  11.384687
lnmean_career_age                             0.033050   0.012852   2.571617
lnex_ld_avg_avgimpact                        -0.264828   0.007259 -36.483771
lnex_ld_avg_insthindex                       -0.049529   0.007468  -6.

In [91]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model4 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model4)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.099926   0.011245   8.886446
CoTypeParticipation                           0.010299   0.015044   0.684582
lnnum_author                                 -0.087020   0.007328 -11.874570
internationalinternational                   -0.045118   0.009576  -4.711696
lnnum_reference                               0.059057   0.008633   6.841129
num_fac                                       0.062924   0.006715   9.369983
SDGTrue                                       0.093974   0.008128  11.561174
lnmean_career_age                             0.012005   0.013072   0.918354
lnex_ld_avg_avgimpact                        -0.269872   0.007355 -36.690437
lnex_ld_avg_insthindex                       -0.051397   0.007510  -6.

In [92]:
# 直接把 4 个模型并排放在一起
tab_latex <- etable(model1, model2, model3, model4,
                    keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                    se = "hetero",
                    tex = TRUE,
                    digits = 3,
                    fitstat = ~ n + r2 + ar2) # 可选：指定要在底部报告的统计量（如样本量、R方、调整R方）

# 打印出可以直接复制到 LaTeX 的代码
cat(tab_latex)

\begingroup \centering \begin{tabular}{lcccc}    \tabularnewline \midrule \midrule    Dependent Variable: & \multicolumn{4}{c}{novel\_uzzi\_bin}\\    Model:                                              & (1)            & (2)            & (3)            & (4)\\      \midrule    \emph{Variables}\\    CoTypeCollaboration                                 & 0.168$^{***}$  & 0.176$^{***}$  & 0.144$^{***}$  & 0.100$^{***}$\\                                                           & (0.011)        & (0.011)        & (0.011)        & (0.011)\\       CoTypeParticipation                                 & 0.047$^{***}$  & 0.095$^{***}$  & 0.047$^{***}$  & 0.010\\                                                           & (0.014)        & (0.015)        & (0.015)        & (0.015)\\       Arts.and.Humanities                                 & 1.84$^{***}$   & 1.81$^{***}$   & 1.76$^{***}$   & 1.74$^{***}$\\                                                           & (0.072)        & (0.072)        